# **<u>Building Your Own AI Chatbot with Streamlit and HKUST GenAI</u>**

**A hands-on guide for ISOM5160**

---

> **Note on this notebook:** This `.ipynb` file is a *reading guide* with a reference code cell. The actual chatbot must be saved as a separate `Chatbot.py` file and run from the terminal with `streamlit run Chatbot.py`. Streamlit apps do not run inside notebook cells.

## **1. Introduction: What Is a Chatbot, and Why Does the LLM Change Everything?**

### *1.1 Two Generations of Chatbots*

Before large language models (LLMs), most chatbots were **rule-driven**. They matched user input against pre-written keywords or decision trees and returned a canned answer. They were fast and predictable, but they could only handle interactions someone had explicitly anticipated.

LLM-powered chatbots work differently. They do **not** retrieve an answer from a database. Instead, they generate a response **token by token**, where each token is predicted based on all the text in the current conversation. This is called **autoregressive generation**. It explains why the same question can produce different answers, and why the model can respond sensibly to phrasing nobody programmed.

**Key idea:** A chatbot is not "smart" because it memorized answers. It is smart because it learned the statistical patterns of language and can generate a reasonable continuation given the context.

### *1.2 Why Streamlit?*

Streamlit turns a plain Python script into an interactive web app in minutes — no HTML, CSS, or JavaScript required. For a chatbot, Streamlit gives you three essential tools:

- `st.chat_input` — the input box at the bottom of the page.
- `st.chat_message` — renders messages as chat bubbles with user/assistant styling.
- `st.session_state` — stores conversation history across page reruns.

That last point matters: Streamlit reruns your entire script on every interaction. Without `st.session_state`, your chatbot would forget the conversation after every message.

### *1.3 Prompt Engineering: Getting the AI to Write Your Chatbot*

#### *1.3.1 What Is Prompt Engineering?*

A **prompt** is the instruction you give an LLM. **Prompt engineering** is the practice of writing that instruction so the model produces useful, correct, and usable output on the first or second try. The model does not read your mind — it only knows what your prompt says (plus whatever general knowledge it learned during training). Vague prompts produce vague code; specific prompts produce specific code.

Four principles matter most for code-generation prompts:

1. **State the role and goal explicitly.** Tell the model *what it is* ("You are a Python developer familiar with Streamlit") and *what the deliverable is* ("a complete `Chatbot.py`").
2. **Specify the constraints.** Name the exact libraries, the API endpoint, the model name, the file structure, and what must *not* appear in the code (e.g., hardcoded keys).
3. **Describe the expected behavior.** Walk through the interaction: what the user sees, what happens on each message, how history is preserved.
4. **Ask for the output format you want.** "Return only the contents of `Chatbot.py` in a single code block" prevents the model from padding the answer with prose you have to delete.

A useful mental model: **treat the prompt like a specification you would hand to a junior developer.** If a competent developer could not build the thing from your prompt alone, the model probably cannot either.

#### *1.3.2 A Worked Example Prompt*

Here is a prompt you can paste into an AI assistant (ChatGPT, Claude, Gemini, or the HKUST GenAI chat interface) to generate a simple chatbot wired to the HKUST GenAI API. Notice what it does **not** do: it does not tell the AI which Streamlit functions to use. It describes the **behavior** you want and lets the model choose the implementation. That is the right division of labor — you know what the app should do, the AI knows the library.

```text
You are a Python developer wanting to build a simple Python chatbot using Streamlit.

Build a single file, Chatbot.py, for a simple multi-turn chatbot that talks to
the HKUST GenAI API (an OpenAI-compatible endpoint).

What the app should do:
- Show a chat interface where the user can type a message and see the reply.
- Remember the conversation, so follow-up questions make sense.
- Use the model "gpt-5.6-sol".
- Ask the user for their own HKUST GenAI API key in a sidebar password field.
  Never write the key directly in the code.
- Connect to base_url="https://hkust.azure-api.net/hkust-genai/v1" and send
  the key as an "api-key" header.
- Tell the user to enter a key before submitting a request.

Output format:
- Return ONLY the complete contents of Chatbot.py in a single ```python code block.
- No explanation before or after the code.
```

Why this prompt works:

- **Role + goal** are stated in the first two lines.
- **Behavior, not methods.** It says "show a chat interface," "remember the conversation," and "ask for a key" — the AI decides how to implement the interface and state. Students do not need to know specific function names in advance.
- **Only the non-negotiables are pinned down:** the package, the model, the sidebar key input, the base URL, and the header. These are project-specific facts the AI cannot guess — everything else is left to the model.
- **Output format** is explicit, so you get copy-pasteable code instead of a tutorial.

#### *1.3.3 What the AI Returns — and How It Maps to Appendix A*

A good response to the prompt above should look almost identical to **Appendix A**. Compare the two by behavior:

| What the prompt asked for | How Appendix A implements it |
|---|---|
| A chat interface the user can type into | `st.chat_input(...)` plus the `st.chat_message` bubbles |
| Remembering the conversation | `st.session_state.messages`, appended on every turn |
| Using the model | `model="gpt-5.6-sol"` in the API call |
| User-supplied key, never hardcoded | `st.text_input(..., type="password")` in the sidebar |
| HKUST base URL + `api-key` header | The `OpenAI(base_url=..., default_headers=...)` block |
| Requiring a key | `st.info(...)` and `st.stop()` when the field is empty |

The AI may choose **different methods** to achieve the same behavior — for example, a different way of laying out the page, or a helper function around the API call. That is normal and instructive: what matters is whether the *behavior* matches the prompt. Use Appendix A as your reference implementation, not as the only acceptable answer.

#### *1.3.4 Iterating on the Prompt*

Real prompt engineering is iterative. If the first response is wrong, do not rewrite the whole prompt — **add one constraint that targets the specific problem**:

- If the model hardcoded the key → add: *"Ask the user for their own key in a sidebar password field; never hardcode it."*
- If the chatbot forgot earlier messages → add: *"The conversation history must be sent with every request so follow-up questions work."*
- If it added files you did not ask for → add: *"Do not include any file other than Chatbot.py."*

This loop — **run, observe, add a targeted constraint, rerun** — is the core skill. The prompt in 1.3.2 is a strong starting point, not a final answer.

#### *1.3.5 Try It Yourself*

Before moving on, do this once:

1. Paste the prompt from 1.3.2 into an AI assistant.
2. Save the returned code as `Chatbot.py` in a scratch folder.
3. Open **Appendix A** next to it and note any differences.

You now have both a working `Chatbot.py` and a feel for how prompt engineering turns a vague request into a deployable program. The rest of this guide (Sections 2–4) uses Appendix A as the reference code and walks through the local setup and deployment steps.

## **2. What the Code Does (Section by Section)**

The complete `Chatbot.py` is provided separately in Appendix A. Here is what each part is doing and why.

### *2.1 Secure API Configuration*

```python
with st.sidebar:
    hkust_api_key = st.text_input("HKUST GenAI API Key", key="hkust_genai_api_key", type="password")
```

The user enters their HKUST GenAI key rather than having it written in the script. The `OpenAI` client below comes from the OpenAI-compatible Python SDK; its `base_url` points to HKUST GenAI, so this app does not require an OpenAI account or key. A password field masks the key on screen; only enter it into an app and hosting environment you trust.

```python
client = OpenAI(base_url="https://hkust.azure-api.net/hkust-genai/v1",
                api_key=hkust_api_key,
                default_headers={"api-key": hkust_api_key})
```

`base_url` points to the HKUST GenAI endpoint, and the HKUST key is also sent in the `api-key` header.

### *2.2 Giving the Chatbot Memory*

```python
if "messages" not in st.session_state:
    st.session_state["messages"] = [{"role": "assistant", "content": "How can I help you?"}]
```

`st.session_state` is a per-session storage box that survives page reruns. Without it, the message list would reset after every interaction.

### *2.3 Displaying the Conversation*

```python
for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])
```

On every rerun, the saved history is redrawn. `st.chat_message` accepts `"user"` and `"assistant"` roles and styles them differently.

### *2.4 Handling Input and Calling the API*

```python
if prompt := st.chat_input():
    if not hkust_api_key:
        st.info("Please add your HKUST GenAI API key to continue.")
        st.stop()
    # Create the client as shown above, then:
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    response = client.chat.completions.create(model="gpt-5.6-sol", messages=st.session_state.messages)
```

- The `:=` operator assigns and checks the value in one step.
- When the user sends a message, the code appends it to history and sends **the entire history** to HKUST. This is how multi-turn conversation works: the model does not remember previous turns on its own.
- The script does not catch API errors, so a failed request can appear as an app error.
- Avoid entering sensitive conversation content unless it is appropriate to send it to HKUST.


## **3. Step-by-Step Local Setup**

### *3.1 Prerequisites*

- Python 3.12 installed.
- An API key from the HKUST ITSC portal (subscribe to the GenAI service first).
- A GitHub account.
- A Streamlit Community Cloud account (sign in with GitHub at share.streamlit.io).

### *3.2 Create the Project Folder and Install Packages*

Open Terminal (Mac) or Command Prompt or Anaconda Prompt (Windows), activate your virtual environment, then create a project folder:

```bash
mkdir hkust-chatbot
cd hkust-chatbot
```

Once done, navigate to VS Code and open the folder in VS Code (Go to `File` -> `Open Folder` -> `hkust-chatbot`). Copy the repository's `Chatbot.py` and `requirements.txt` into this folder. Appendix A also shows the chatbot code.

The repository's `requirements.txt` contains the Streamlit app dependencies. The `openai` package is used as the client SDK for HKUST's OpenAI-compatible endpoint; it does not require an OpenAI API key.

```text
streamlit>=1.28
openai>=1.2
```

After saving `requirements.txt`, install the packages from that file by running the following command on Terminal (Mac) or Command Prompt or Anaconda Prompt (Windows):

```bash
pip install -r requirements.txt
```

Using `requirements.txt` means the same dependency list can later be reused by Streamlit Community Cloud during deployment.

### *3.3 Enter Your API Key*

This version of `Chatbot.py` asks each user to enter their own HKUST GenAI API key in the sidebar password field.

Never hardcode your key or upload it to GitHub. A password field masks the key on screen, but the app receives it, so only use an app and hosting environment you trust.

### *3.4 Test Locally*

```bash
streamlit run Chatbot.py
```

A browser window should open. Enter your HKUST key in the sidebar. If the chatbot responds to messages, your local setup is correct. You are now ready to deploy.

## **4. Deploying to GitHub and Streamlit Community Cloud**

### *4.1 Create the Repository*

1. Go to `github.com` and sign in.
2. Click **+** (top right) → **New repository**.
3. Name it, e.g., `hkust-chatbot`. Keep it **Public** (required for free Streamlit Cloud deployment).
4. Click **Create repository**.

### *4.2 Upload Your Files Through the GitHub Website*

Now upload your two code files through the browser — no commands needed.

1. On your repository's main page, click **Add file** → **Upload files**.
2. Drag and drop `Chatbot.py` and `requirements.txt` from your project folder into the browser window.
3. Scroll down, enter a short message such as `Add chatbot files`, and click **Commit changes**.

Your repository now contains `Chatbot.py` and `requirements.txt`. Confirm that no API keys were uploaded.

### *4.3 Deploy on Streamlit Community Cloud*

1. Go to share.streamlit.io and sign in with GitHub.
2. Click **Create app**.
3. When asked "Do you already have an app?", choose **Yup, I have an app**.
4. Fill in:
   - **Repository:** `YOUR_USERNAME/YOUR_REPO_NAME`
   - **Branch:** `main`
   - **Main file path:** `Chatbot.py`
5. Each user enters their own HKUST GenAI key in the app's sidebar after deployment.
6. Click **Deploy**.

The build usually takes 1–3 minutes. Once live, test the chatbot in the browser.

Only enter a key or conversation content into a hosted app and environment you trust. The hosting operator may be able to access information submitted to the app.

## **5. Troubleshooting**

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'openai'` | Missing install in the active environment | Activate the intended environment and run `pip install -r requirements.txt` |
| Authentication or request error | Invalid key, missing subscription, model access, network issue, or service outage | Check your HKUST key, subscription, connection, and service availability |
| Prompt asks for a key | Sidebar field is empty | Enter your HKUST GenAI API key in the sidebar |
| `ModuleNotFoundError` in the cloud only | `requirements.txt` missing from the repository | Upload `requirements.txt` and redeploy |
| Changes not showing after upload | Cloud app not rebuilt | Check the app's build and reboot it if needed |
| Chat history disappears | A new or reset browser session | Start a new conversation; session history is not permanent |
| An API key appears on GitHub | It was accidentally uploaded | Remove the exposed key and rotate it |

The current script does not catch API errors. It appends the user message before the request, so a failed message remains in session history.


## **6. Key Takeaways**

- A chatbot's "intelligence" comes from **autoregressive token prediction**, not from a lookup table.
- **Prompt engineering** turns a vague request into a deployable program: state the role and goal, pin down constraints, describe behavior, and specify the output format.
- Multi-turn conversation requires **resending the full history** with every API call.
- `st.session_state` is what makes Streamlit apps remember anything between reruns.
- **Never hardcode API keys.** This app asks each user for their HKUST key in a sidebar password field.

## **Appendix A: The Complete `Chatbot.py`**

Save the following as a separate file named `Chatbot.py` in your project folder. Do **not** run it inside a notebook cell — run it from the terminal with `streamlit run Chatbot.py`.

In [ ]:
from openai import OpenAI
import streamlit as st

with st.sidebar:
    hkust_api_key = st.text_input("HKUST GenAI API Key", key="hkust_genai_api_key", type="password")
    st.caption("Enter your HKUST GenAI API key.")
    "[View the source code](https://github.com/1HD22/ISOM5160-Chatbot-Example)"

st.title("💬 Chatbot")
st.caption("🚀 A Streamlit chatbot powered by HKUST GenAI")
if "messages" not in st.session_state:
    st.session_state["messages"] = [{"role": "assistant", "content": "How can I help you?"}]

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input():
    if not hkust_api_key:
        st.info("Please add your HKUST GenAI API key to continue.")
        st.stop()

    client = OpenAI(base_url="https://hkust.azure-api.net/hkust-genai/v1",
                    api_key=hkust_api_key,
                    default_headers={"api-key": hkust_api_key})
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    response = client.chat.completions.create(model="gpt-5.6-sol", messages=st.session_state.messages)
    msg = response.choices[0].message.content
    st.session_state.messages.append({"role": "assistant", "content": msg})
    st.chat_message("assistant").write(msg)
